# UAVIDS-2025 — auditoria reproduzível e protocolos candidatos

**Papel deste notebook:** estabelecer identidade do arquivo, contrato provisório dos atributos e divisões candidatas antes de ajustar classificadores. Este notebook não compara modelos e não produz uma conclusão de superioridade.

O código está em inglês para facilitar manutenção e revisão. As decisões, interpretações e limites são escritos em português. A lógica reutilizável fica em `src/uavids_study/data_audit.py`; o notebook funciona como narrativa executável e não como fonte oculta de regras experimentais.


## Decisões científicas anteriores aos resultados

1. A tarefa principal tem cinco classes e a unidade é um registro de fluxo.
2. `FlowID`, `SrcAddr` e `DstAddr` não entram como preditores no painel principal; os endereços permanecem disponíveis para grupos e diagnóstico. `Protocol` é removido se sua constância for confirmada.
3. S0 mede interpolação na mistura empírica; S1 mantém assinaturas numéricas idênticas no mesmo fold; S2 mantém origens inteiras no mesmo fold. Esses protocolos representam populações distintas.
4. A ausência de timestamp, sessão e execução impede tratar a ordem do CSV ou o `FlowID` como cronologia validada.
5. A seleção de modelos futura deverá usar pipelines ajustados somente no treino e validação interna. O teste externo não orientará preprocessing, hiperparâmetros ou limiares.

**Inspiração documentada.** O artigo do dataset define o objeto de estudo ([Zeng et al., IEEE CNS 2025](https://doi.org/10.1109/CNS66487.2025.11194990)). A separação de grupos e a prevenção de ajuste fora dos folds seguem a documentação de [validação cruzada](https://scikit-learn.org/stable/modules/cross_validation.html) e [erros comuns](https://scikit-learn.org/stable/common_pitfalls.html) do scikit-learn. R7 motiva testar explicitamente o efeito de `FlowID` porque seu texto/figura deixam ambígua a presença do identificador ([artigo](https://doi.org/10.1186/s13635-026-00234-w)). R2 motiva contexto temporal/relacional, mas o CSV público não contém sozinho os metadados temporais necessários ([FedGraph-ID](https://doi.org/10.1109/INFOCOM59046.2026.11571676)).


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "research":
    project_root = project_root.parents[1]
elif project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

from uavids_study.data_audit import run_data_audit

dataset_path = project_root / "notebooks" / "data" / "raw" / "UAVIDS-2025.csv"
artifact_dir = project_root / "research_artifacts" / "data_audit"

print(f"Project root: {project_root}")
print(f"Dataset exists: {dataset_path.exists()}")


Project root: C:\Users\User\Documents\projects\desafio-UAV
Dataset exists: True


## 1. Identidade do artefato

O hash é um requisito científico: dois arquivos com o mesmo nome podem representar versões ou ordenações diferentes. A execução abaixo falha deliberadamente se o CSV local não corresponder ao artefato registrado como Zenodo v1. Os dados brutos não são modificados.


In [2]:
created_paths, audit_summary = run_data_audit(dataset_path, artifact_dir)
data_manifest = json.loads(created_paths.manifest.read_text(encoding="utf-8"))

identity = data_manifest["dataset"]
print(json.dumps(identity, indent=2, ensure_ascii=False))


{
  "path_recorded_as": "UAVIDS-2025.csv",
  "bytes": 20054667,
  "rows": 122171,
  "columns": 23,
  "column_names": [
    "FlowID",
    "FlowDuration/s",
    "SrcAddr",
    "SrcPort",
    "DstAddr",
    "DstPort",
    "Protocol",
    "TxPackets",
    "RxPackets",
    "LostPackets",
    "TxBytes",
    "RxBytes",
    "TxPacketRate/s",
    "RxPacketRate/s",
    "TxByteRate/s",
    "RxByteRate/s",
    "MeanDelay/s",
    "MeanJitter/s",
    "Throughput/Kbps",
    "MeanPacketSize",
    "PacketDropRate",
    "AverageHopCount",
    "label"
  ],
  "md5": "ec84ed5390d5de42b07e8a011709ff82",
  "sha256": "d50d339f68be7b23f0bf089dd438b20a1835c13182d8641220538121440164d0",
  "exact_match_to_registered_zenodo_v1": true
}


## 2. Integridade, classes e contrato dos atributos

As definições do dicionário são provisórias. Relações observadas no CSV ajudam a detectar redundância, mas não substituem a fórmula oficial nem comprovam que uma variável estaria disponível causalmente em um UAV real.


In [3]:
class_distribution = pd.read_csv(created_paths.class_distribution)
data_dictionary = pd.read_csv(created_paths.dictionary)

print(class_distribution.to_string(index=False))
print("\nResumo da integridade:")
print(json.dumps(audit_summary, indent=2, ensure_ascii=False))


           label  rows  fraction
  Normal Traffic 26172  0.214224
Blackhole Attack 26110  0.213717
 Wormhole Attack 26086  0.213520
    Sybil Attack 24077  0.197076
 Flooding Attack 19726  0.161462

Resumo da integridade:
{
  "rows": 122171,
  "columns": 23,
  "classes": 5,
  "missing_cells": 0,
  "infinite_numeric_cells": 0,
  "constant_columns": [
    "Protocol"
  ],
  "full_duplicate_rows_after_first": 0,
  "unique_model_signatures": 113952,
  "duplicate_model_signature_rows_after_first": 8219,
  "duplicate_signature_and_label_rows_after_first": 8219,
  "rows_belonging_to_repeated_model_signatures": 10774,
  "conflicting_model_signatures": 0,
  "packet_drop_rate_above_one": 81,
  "packet_drop_rate_below_zero": 0,
  "flow_id_unique": true,
  "flow_id_sequential_in_file_order": true,
  "adjacent_same_label_fraction": 0.9299336989440943,
  "source_addresses": 176,
  "destination_addresses": 218,
  "protocol_values": [
    "UDP"
  ],
  "interpretation_limits": [
    "The public CSV has 

In [4]:
selected_columns = [
    "column", "dtype", "role", "unit", "formula",
    "unique_values", "missing_values", "available_at_prediction_time",
]
print(data_dictionary[selected_columns].to_string(index=False))


         column   dtype                role      unit                                            formula  unique_values  missing_values available_at_prediction_time
         FlowID   int64          identifier       NaN                                      not confirmed         122171               0                           no
 FlowDuration/s float64     numeric_feature         s                                      not confirmed          62227               0                  unconfirmed
        SrcAddr  object relational_metadata       NaN                                      not confirmed            176               0                  unconfirmed
        SrcPort   int64     numeric_feature      port                                      not confirmed              2               0                  unconfirmed
        DstAddr  object relational_metadata       NaN                                      not confirmed            218               0                  unconfirmed
        Ds

## 3. Repetições, ordem e relações algébricas

Duplicatas completas incluem identificadores e podem ocultar repetições do vetor que chega ao modelo. Por isso, a assinatura S1 usa as 18 variáveis numéricas do painel principal. Assinaturas repetidas não são apagadas: sua frequência faz parte da população empírica e todas permanecem no mesmo fold em S1.


In [5]:
duplicates_by_class = pd.read_csv(created_paths.duplicates_by_class)
algebraic_relations = pd.read_csv(created_paths.algebraic_relations)

print(duplicates_by_class.to_string(index=False))
print("\nRelações algébricas auditadas:")
print(algebraic_relations.to_string(index=False))


           label  rows  rows_in_repeated_signature  fraction
Blackhole Attack 26110                           0  0.000000
 Flooding Attack 19726                       10462  0.530366
  Normal Traffic 26172                           0  0.000000
    Sybil Attack 24077                         312  0.012958
 Wormhole Attack 26086                           0  0.000000

Relações algébricas auditadas:
                                    relation  comparable_rows  close_rows  close_fraction  mean_absolute_error  max_absolute_error    rtol         atol
   PacketDropRate ~= LostPackets / TxPackets           122171      122148        0.999812         1.585740e-07            0.000005 0.00001 1.000000e-08
TxPacketRate/s ~= TxPackets / FlowDuration/s           122171      122168        0.999975         1.814197e-04           15.119160 0.00001 1.000000e-08
RxPacketRate/s ~= RxPackets / FlowDuration/s           122171      122168        0.999975         9.691268e-05            7.559580 0.00001 1.00000

In [6]:
endpoint_profile = pd.read_csv(created_paths.class_endpoint_profile)
feature_ranges = pd.read_csv(created_paths.feature_ranges)

print(endpoint_profile.to_string(index=False))
print("\nFaixas observadas (não são domínios físicos validados):")
print(feature_ranges.to_string(index=False))


           label  rows  source_addresses  destination_addresses  source_ports  destination_ports  minimum_flow_id  maximum_flow_id
Blackhole Attack 26110               110                    110             2                  2            19181            94285
 Flooding Attack 19726               120                    200             1                  1            50580           122171
  Normal Traffic 26172                95                    100             2                  2            57961            94283
    Sybil Attack 24077                37                    108             1                  1                1           109046
 Wormhole Attack 26086               150                    150             2                  2            35138           108071

Faixas observadas (não são domínios físicos validados):
         column   minimum       median      maximum  zero_rows  negative_rows
         FlowID  1.000000 61086.000000 1.221710e+05          0              0
 

## 4. Protocolos candidatos S0, S1 e S2

Os folds abaixo são persistidos em `split_candidates.csv.gz`, portanto modelos diferentes receberão exatamente as mesmas linhas. Eles ainda são candidatos: S2 apresenta grupos grandes e precisa ser revisado antes do congelamento final. `groups_crossing_folds = 0` é uma condição necessária para S1/S2, mas não garante ausência de outras formas de dependência.


In [7]:
protocol_diagnostics = pd.read_csv(created_paths.protocol_diagnostics)
group_diagnostics = pd.read_csv(created_paths.group_diagnostics)
fold_balance = pd.read_csv(created_paths.fold_balance)

print(protocol_diagnostics.to_string(index=False))
print("\nTamanho e pureza dos grupos:")
print(group_diagnostics.to_string(index=False))
print("\nTotal de linhas por fold:")
print(fold_balance.groupby(["protocol", "fold"])["rows"].sum().to_string())


protocol group_constraint   rows  groups  groups_crossing_folds  maximum_folds_per_group  minimum_fold_rows  maximum_fold_rows
      S0        signature 122171  113952                   2349                        5              24434              24435
      S1        signature 122171  113952                      0                        1              24352              24550
      S2           source 122171     176                      0                        1              16071              30585

Tamanho e pureza dos grupos:
       group_type  groups  minimum_rows  median_rows  mean_rows  maximum_rows  groups_with_multiple_labels
numeric_signature  113952             1          1.0   1.072127            24                            0
   source_address     176             1        530.0 694.153409          3341                          125

Total de linhas por fold:
protocol  fold
S0        0       24435
          1       24434
          2       24434
          3       24434
   

In [8]:
overlap_diagnostics = pd.read_csv(created_paths.fold_overlap_diagnostics)
print(overlap_diagnostics.to_string(index=False))


protocol  fold              entity  train_unique  test_unique  overlap_unique  test_overlap_fraction
      S0     0      source_address           174          176             174               0.988636
      S0     0 destination_address           218          202             202               1.000000
      S0     0   numeric_signature         91590        23671            1309               0.055300
      S0     1      source_address           176          174             174               1.000000
      S0     1 destination_address           216          202             200               0.990099
      S0     1   numeric_signature         91625        23671            1344               0.056778
      S0     2      source_address           176          174             174               1.000000
      S0     2 destination_address           218          205             205               1.000000
      S0     2   numeric_signature         91623        23639            1310              

## 5. Leitura dos achados e decisões resultantes

- O arquivo local corresponde exatamente ao Zenodo v1 por tamanho, número de linhas/colunas, MD5 e SHA-256.
- Há 0 valores ausentes e 0 infinitos. `Protocol` é constante (`UDP`) e será removido do painel principal.
- Existem 8,219 repetições além da primeira entre 113,952 assinaturas numéricas únicas. 53.04% dos registros de Flooding pertencem a uma assinatura repetida. Como não há conflitos de rótulo entre assinaturas idênticas, S1 consegue mantê-las juntas sem resolver rótulos contraditórios.
- A relação `PacketDropRate ≈ LostPackets / TxPackets` vale dentro da tolerância para 99.9812% das linhas comparáveis. Ainda assim, 81 registros têm `PacketDropRate > 1`; eles serão preservados até a fórmula/semântica ser confirmada.
- `FlowID` é único e sequencial, e 92.99% dos pares adjacentes mantêm o mesmo rótulo. Isso torna uma divisão sem embaralhamento inadequada e não comprova cronologia.
- S2 preserva as origens, mas os tamanhos dos folds são [23293, 24144, 28078, 16071, 30585]. A diferença decorre de apenas 176 origens com tamanhos desiguais. A métrica deverá ser agregada por predição e também apresentada por fold, com intervalos e distribuição visíveis.
- S2 não elimina automaticamente destinos ou assinaturas já vistos. O arquivo de sobreposição quantifica esse limite em cada fold.
- O CSV não oferece timestamp, sessão, cenário ou execução. O eixo temporal permanece bloqueado; uma GNN temporal reproduzível exige metadados/código adicionais de R2.

Esses achados justificam comparar o protocolo aleatório usado por muitos trabalhos com S1 e S2. Eles ainda não demonstram que um modelo publicado sofre vazamento, nem que uma arquitetura é superior.


## 6. Critérios para iniciar o notebook de modelagem

- [x] Arquivo identificado e imutável por hash.
- [x] Classes, ausentes, infinitos, constantes, faixas, repetições e relações algébricas auditados.
- [x] `FlowID` e endereços separados dos atributos principais.
- [x] S0/S1/S2 materializados uma única vez, com grupos e sobreposições verificáveis.
- [x] Exposição prévia e limites de inferência registrados em `protocol/scope.md`.
- [ ] Revisão humana do dicionário e das alegações atribuídas aos artigos.
- [ ] Confirmação da licença exibida pela fonte canônica.
- [ ] Orçamento computacional, hardware e publicação-alvo registrados.
- [ ] Estratégia de validação interna e espaços de busca congelados antes do primeiro tuning comparativo.

O próximo notebook deve implementar um baseline ingênuo e os modelos tabulares em pipelines, ler os folds deste notebook e salvar predições por linha. A reprodução com `FlowID` deve ser rotulada como diagnóstico/reprodução fiel e nunca misturada ao resultado principal corrigido.
